In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [24]:
train_data_transform = transforms.Compose([
      transforms.Resize((128, 128)),
      transforms.RandomHorizontalFlip(),
      transforms.RandomRotation(10),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_data_transform = transforms.Compose([
      transforms.Resize((128, 128)),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [25]:
train_dataset = datasets.ImageFolder("./train", transform=train_data_transform)
test_dataset = datasets.ImageFolder("./test", transform=test_data_transform)

In [26]:
train_data_loader = DataLoader(train_dataset, shuffle=True, batch_size=32)
test_data_loader = DataLoader(test_dataset, shuffle=True, batch_size=32)

In [33]:
class CNeuralNetwork(nn.Module):
      
      def __init__(self):
            super().__init__()

            self.features = nn.Sequential(
                  nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding="same"),
                  nn.ReLU(),
                  nn.MaxPool2d(kernel_size=2, stride=2),

                  nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding='same'),
                  nn.ReLU(),
                  nn.MaxPool2d(kernel_size=2, stride=2)
            )
            self.fc = nn.Sequential(
                  nn.Flatten(),
                  nn.Linear(64 * 32 * 32, 128),
                  nn.ReLU(),
                  nn.Linear(128, 64),
                  nn.ReLU(),
                  nn.Linear(64, 2)
            )
      

      def forward(self, x):
            X = self.features(x)
            return self.fc(X)

In [34]:
model = CNeuralNetwork()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

In [ ]:
for epoch in range(epochs):

      total_epochs_loss = 0
      for image, labels in train_data_loader:
            output = model(image)

            loss = criterion(output, labels)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            total_epochs_loss += loss.item()
      avg_loss = total_epochs_loss/len(train_data_loader)
      print(f"Epochs: {epoch + 1}, loss: {avg_loss:.4f}")